# Stronger Model: Gradient Boosting

In this notebook, I will test a stronger machine learning model than the Logistic Regression baseline.

The baseline ROC-AUC was **0.938**. The goal here is to see whether a model that can learn more complex patterns can improve this score.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score


## Load the Data

I will load the training data and use the same target and feature setup as the baseline model.


In [2]:
train = pd.read_csv("../data/train.csv")

TARGET = "Will_Buy_EV"
ID_COLUMN = "id"

X = train.drop(columns=[TARGET, ID_COLUMN])
y = train[TARGET].map({"No": 0, "Yes": 1})

print("Training shape:", X.shape)
print("Target shape:", y.shape)


Training shape: (668665, 13)
Target shape: (668665,)


## Prepare the Target and Features

The target is Will_Buy_EV.

I will remove the id column because it is only an identifier and should not be used as a prediction feature.


In [3]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("Training positive rate:", round(y_train.mean(), 4))
print("Validation positive rate:", round(y_valid.mean(), 4))


Training rows: 534932
Validation rows: 133733
Training positive rate: 0.1746
Validation positive rate: 0.1746


## Train-Validation Split

I will use the same 80/20 stratified split from the baseline model.

Keeping the same split makes the model comparison fair because both models are tested on the same validation data.


In [4]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


Numerical features: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Categorical features: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


## Encode Categorical Features

The gradient boosting model needs numerical input.

I will use ordinal encoding for the categorical columns. Unknown categories are handled safely in case a category appears in the validation data that was not seen during training.


In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "encoder",
                    OrdinalEncoder(
                        handle_unknown="use_encoded_value",
                        unknown_value=-1
                    )
                )
            ]),
            categorical_features
        )
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_valid_encoded = preprocessor.transform(X_valid)

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded validation shape:", X_valid_encoded.shape)


Encoded training shape: (534932, 13)
Encoded validation shape: (133733, 13)


## Train the Model

The model used here is HistGradientBoostingClassifier.

Gradient boosting builds many small decision trees one after another. Each new tree tries to improve the mistakes made by the previous trees.

This allows the model to learn more complex relationships than the Logistic Regression baseline.


In [6]:
model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=200,
    max_leaf_nodes=31,
    random_state=42
)

model.fit(X_train_encoded, y_train)

print("Model training complete.")


Model training complete.


## Validation ROC-AUC

The competition uses ROC-AUC, so I will evaluate the model using the same metric.

The main number to compare against is our Logistic Regression baseline of **0.938**.


In [7]:
valid_probabilities = model.predict_proba(X_valid_encoded)[:, 1]

roc_auc = roc_auc_score(y_valid, valid_probabilities)

print(f"Gradient Boosting ROC-AUC: {roc_auc:.3f}")
print("Logistic Regression baseline: 0.938")


Gradient Boosting ROC-AUC: 0.941
Logistic Regression baseline: 0.938


## Compare with the Baseline

If this model scores higher than 0.938, we have evidence that the stronger model is learning useful patterns that Logistic Regression did not capture.

If it does not improve the score, that is still useful because it tells us that we need to look at the features and model setup more carefully.


In [8]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "HistGradientBoosting"
    ],
    "ROC-AUC": [
        0.938,
        roc_auc
    ]
})

results


,Model,ROC-AUC
0,Logistic Regression,0.938000
1,HistGradientBoosting,0.940597


## What's Next

After this model, we will compare the results and decide what to improve next.

We will not tune everything at once. We will make changes one at a time so we can understand what actually improves the score.
